# Usages

## Overview

- This page shows a general pipeline for analyzing LiP data via lipana
- The example data is a truncated search report from DIA-NN, which has 3 experiment conditions and 3 replicates per condition, and only 1000 proteins are kept
- The example files are in "path_to/LiPAna/example_data"

In [ ]:
import gzip
import shutil
from pathlib import Path

workspace = Path(".").resolve().parents[1].joinpath("example_data")
print("current workspace:", str(workspace))
# Unzip gzipped files
for file in workspace.glob("*.gz"):
    with gzip.open(file, "rb") as f_in:
        with open(file.with_suffix(""), "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)


current workspace: /home/lourh/Project-LiPDIA/project_src/LiPAna/example_data


In [2]:
import sys

sys.path.append(str(workspace.parent))
import lipana

In [3]:
from time import sleep

import polars as pl

## Define the experiment layout

- At first, an experiment layout is required for annotating the expected report
- Below will introduce two approaches to do so

1. Load a text file which stores the experiment layout, like the following ("replicate" can be omitted)

|  run | condition | replicate |
| ---: | --------: | --------: |
| run1 |      ctrl |         1 |
| run2 |      ctrl |         2 |
| run3 |      ctrl |         3 |
| run4 |      exp1 |         1 |
| run5 |      exp1 |         2 |
| run6 |      exp1 |         3 |
| run7 |      exp2 |         1 |
| run8 |      exp2 |         2 |


In [4]:
import lipana

exp_layout = lipana.ExperimentLayout.from_file(workspace.joinpath("experiment_layout.txt"))
print("The input file is:")
exp_layout.exp_df

The input file is:


run,condition,replicate
str,str,i64
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",1
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",2
"""20240404_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",3
"""20240331_YSJ-HY_5_100-400ng_A_…","""H5_Y100""",1
"""20240331_YSJ-HY_5_100-400ng_B_…","""H5_Y100""",2
"""20240404_YSJ-HY_5_100-400ng_C_…","""H5_Y100""",3
"""20240331_YSJ-HY_25_100-400ng_A…","""H25_Y100""",1
"""20240331_YSJ-HY_25_100-400ng_B…","""H25_Y100""",2
"""20240404_YSJ-HY_25_100-400ng_C…","""H25_Y100""",3


2. Init by passing a run-to-condition map

In [5]:
import lipana

exp_layout = lipana.ExperimentLayout.from_run_to_condition_map(
    {
        "20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154": "H2d5_Y100",
        "20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155": "H2d5_Y100",
        "20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206": "H2d5_Y100",
        "20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157": "H5_Y100",
        "20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158": "H5_Y100",
        "20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210": "H5_Y100",
        "20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163": "H25_Y100",
        "20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164": "H25_Y100",
        "20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220": "H25_Y100",
    }
)

print("This results in the same experiment layout as the previous example, because replicate is incremented from 1")
exp_layout.exp_df

This results in the same experiment layout as the previous example, because replicate is incremented from 1


run,condition,replicate
str,str,i64
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",1
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",2
"""20240404_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",3
"""20240331_YSJ-HY_5_100-400ng_A_…","""H5_Y100""",1
"""20240331_YSJ-HY_5_100-400ng_B_…","""H5_Y100""",2
"""20240404_YSJ-HY_5_100-400ng_C_…","""H5_Y100""",3
"""20240331_YSJ-HY_25_100-400ng_A…","""H25_Y100""",1
"""20240331_YSJ-HY_25_100-400ng_B…","""H25_Y100""",2
"""20240404_YSJ-HY_25_100-400ng_C…","""H25_Y100""",3


## Load a search report

- In this step, the example report will be loaded and annotated
  - A DIA-NN report can be from either executing DIA-NN alone or from the FragPipe that uses DIA-NN to process DIA data
  - For Spectronaut report, there are some required columns and a template of report setting can be generated by the following command: `lipana.report.export_sn_report_setting("path_to_store_the_setting_file.rs")`

### First load FASTA file(s)

- FASTA files should be first loaded and parsed for further use for report annotation
- `lipana.parse_fasta` will read one or more FASTA files and return a `ParsedFasta` object
  - `fasta_path` can be a path points to a FASTA or a list of paths (`fasta_path=some_path` or `fasta_path=[path1, path2]`)
  - `contam_fasta_path`: this parameter will be useful to strictly exclude potential contaminants. The species of any protein entries in the given file will be annotated as "Contam" instead of its original species, including those presented in `fasta_path`
  - `contaminations`: a list of additional protein entries can be given here, and this parameter can also work alone if `contam_fasta_path` is `None`

In [6]:
parsed_fasta = lipana.parse_fasta(
    fasta_path=workspace.joinpath("human_yeast_contam.fasta"),
    contam_fasta_path=workspace.joinpath("contam.fasta"),
    contaminations=None,
    gen_species_to_concat_seqs=True,  # This parameter is useful to annotate species of peptides via a whole protein sequence map, but is time-consuming
    workspace=workspace,
    resume=True,
    write_parsed_fasta=True,
)

### Load and annotate a search report

- In this step, a search report will be loaded and annotated via the already initialized `ExperimentLayout` and `ParsedFasta`

In [7]:
# Below two methods have the same paramters
# lipana.DIANNReport.load_search_report(...)
# lipana.SpectronautReport.load_search_report(...)

# Here will load a DIA-NN search report as an example, and show a subset of available parameters
report = lipana.DIANNReport.load_search_report(
    workspace.joinpath("example_diann_report.txt"),
    exp_layout=exp_layout,
    parsed_fasta=parsed_fasta,
    do_species_annotation=True,  # Set this to True if the experiment has multiple species (and they should also be presented in FASTA files)
    pre_annotation_filter=(
        (pl.col("Q.Value") < 0.01)
        & (pl.col("Lib.PG.Q.Value") < 0.01)
        & (pl.col("Protein.Group").is_not_null())
        & (pl.col("Precursor.Quantity").is_not_null())
        & (pl.col("Precursor.Quantity") > 1.1)
    ),  # The filters defined here will be applied to the report after loading immediately, and here the 5 rules are default for DIA-NN (if this parameter is not given, these rules will be applied); for Spectronaut, they are (
    #     (pl.col("PG.ProteinGroups").is_not_null())
    #     & (pl.col("FG.Quantity").is_not_null())
    #     & (pl.col("FG.Quantity") > 1.1)
    # )
    post_annotation_filter=None,  # This parameter can be used to filter the report after annotation
    restricted_cut_sites=("K", "R"),
    expand_to_cut_site_level=True,  # This will make each cut site one row, which means each precursor row will be duplicated
    # modification_map=None,  # This parameter can be either None or a map from software used modification to UniMod. By default, DIA-NN report has None because it use UniMod as default; for `SpectronautReport`, this parameter has a default value as {
    #     "[Acetyl (Protein N-term)]": "(UniMod:1)",
    #     "[Carbamidomethyl (C)]": "(UniMod:4)",
    #     "[Oxidation (M)]": "(UniMod:35)",
    # }
)
# The variable `report` is now a `SearchReport` object, and basic annotations have been done
report

Search report object with main report shape: (113057, 81)
Number of quantification data: 0
Number of statistics results: 0
Workspace: /home/lourh/Project-LiPDIA/project_src/LiPAna/example_data

## Prepare quantification matrices

- A newly generated `SearchReport` object will only store a preprocessed search report and some meta-data
- In this section, some quantification matrices will be added to this object
- The quantification can be just a pivot of the report, or do quantification aggregation via topN or maxLFQ


### Use raw reported quantity values

- This will do a simple conversion to get a quantification matrix by using software reported raw quantity values as cells, and is useful to get a precursor or protein group level matrix
- This step also shows how to attach a customized quantification report to the main report manually via `attach_quant_data` method

In [8]:
quant_data = report.attach_quant_data(
    lipana.convert_long_report_to_wide(
        report,
        index_col="precursor",
        column_col="run",
        value_col="precursor_quantity",
        do_log_scale=2,
        pl_filter=(
            ~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)
        ),  # This parameter filters out those peptides mapped to multiple species
        do_unique=True,
    ),
    entry_level="precursor",  # These two parameters will be the name of this quantification report, and can be used to retrieve it from the main report object
    quant_method="precursor_quantity",
)

# To retrieve this object from the main report object, use the following command
quant_data = report.get_quant_data("precursor", "precursor_quantity")


### maxLFQ or topN

- The maxLFQ and topN methods share the same interface by calling `SearchReport.construct_and_attach_quant_data` method, and have some convenient designs to handle more complicated scenarios
- The following examples will only use "maxlfq", and topN can be configured by setting `method="top3"`, `method="top1"`, ...

#### Peptide matrix from precursor MS2 via maxLFQ

In [9]:
# This will do maxLFQ on precursor quantities to get a peptide level matrix
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_prec",
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",
    low_level_entry_col="precursor",
    base_quant_col="precursor_quantity",
    require_expansion=False,
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmp96yb1iyo.txt
Removing low intensities...

Output to /tmp/tmp96yb1iyo.txt-iq_output.txt



Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmp96yb1iyo.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 55844
# quantitative values after filtering = 55844

# samples  = 9
# proteins = 9577
nrow = 55844, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
15%
20%
25%
30%
35%
41%
46%
52%
58%
64%
69%
74%
80%
85%
90%
Completed.


#### Peptide matrix from prec MS1, prec MS2, and fragments, via maxLFQ

In [10]:
# The quantities of base level entries can also be from multiple sources
# The following example will use maxLFQ to aggregate quantities from three levels: precursor MS1, precursor MS2, and fragments
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_multi",
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",
    low_level_entry_col=("precursor", "precursor", "Fragment.Info"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1", "Fragment.Quant.Raw"),
    require_expansion=(False, False, ";"),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmps9jzxjsm.txt



Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmps9jzxjsm.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 701484
# quantitative values after filtering = 701484

# samples  = 9
# proteins = 9577


Removing low intensities...



nrow = 701484, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
16%
21%
27%
33%
38%
44%
49%
54%
61%
66%
72%
79%
84%
90%
97%
Completed.


Output to /tmp/tmps9jzxjsm.txt-iq_output.txt


#### Non-restricted cut site from precursor MS2

In [11]:
# Sometimes, the quantity values should be aggregated from a filtered report
# Except to directly filter the report, call quantification function, and attach the matrix to main report
# The following example will generate a cut site level matrix by only using semi-specific peptides
quant_data = report.construct_and_attach_quant_data(
    quant_name="nonrestricted_cut_site_by_maxlfq_from_prec",
    method="maxlfq",
    filter_condition=(
        (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
        & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
    ),
    run_col="run",
    primary_entry_col="cut_site",
    low_level_entry_col="precursor",
    base_quant_col="precursor_quantity",
    require_expansion=False,
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)


Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmp0bwzp8nt.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 51767
# quantitative values after filtering = 51767

# samples  = 9
# proteins = 7146
nrow = 51767, # proteins = 7146, # samples = 9
Using 95 threads...
1%
6%
11%
16%
21%
26%
31%
36%
41%
46%
51%
56%
61%
66%
71%
77%
84%
89%
Completed.


MaxLFQ estimation via iq for /tmp/tmp0bwzp8nt.txt
Removing low intensities...

Output to /tmp/tmp0bwzp8nt.txt-iq_output.txt


### Utilities on quantification report

- The quantification matrix generated from the above sections is actually an object `EntryQuantificationReport`, which has many handful functions

In [12]:
# If the generated object is not explicitly assigned to a variable, can use the following way to get from main report object
quant_data = report.get_quant_data("precursor", "precursor_quantity")

# The following three methods will attach new columns to the dataframe stored in the quantification report object
quant_data.calc_cv(cond="all", min_reps=3, temp_reverse_log_scale=2, new_colname_pattern="{cond}_cv_{min_reps}reps")
quant_data.count_detected_replicates(new_colname_pattern="{cond}_detected_reps")
quant_data.calc_ratio(
    base_cond="H5_Y100",
    is_log=True,
    temp_reverse_log_scale=2,
    div_method="agg_and_divide",
    agg_method="mean",
    new_colname_pattern="ratio_{cond1}_to_{cond2}",
)

precursor,20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164,20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163,20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155,20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157,20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220,20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206,20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158,20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210,20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154,H5_Y100_cv_3reps,H25_Y100_cv_3reps,H2d5_Y100_cv_3reps,H5_Y100_detected_reps,H25_Y100_detected_reps,H2d5_Y100_detected_reps,ratio_H25_Y100_to_H5_Y100,ratio_H2d5_Y100_to_H5_Y100
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32
"""LPAEVEIPKNVDIS/2""",14.434139,13.938183,13.895041,13.805732,14.804245,13.591607,14.452115,14.51086,14.174753,24.786989,29.152855,20.031633,3,3,3,0.145257,-0.382955
"""YFQAPSDYR/2""",13.567189,13.716377,null,null,null,null,null,null,null,NaN,NaN,NaN,0,2,0,NaN,NaN
"""KAVSNEELK/2""",17.286247,17.239571,17.427143,17.161185,17.655512,17.314304,17.239937,17.538599,17.595646,14.211181,16.409777,9.89438,3,3,3,0.083509,0.12777
"""LKELPDTVGELR/2""",14.283339,14.581157,null,11.527857,14.280874,null,null,null,null,NaN,12.350394,NaN,1,3,0,2.860971,NaN
"""LYDKTENDIK/2""",16.569562,16.734969,16.672663,16.763497,16.716848,16.731104,16.803318,16.890123,16.953013,4.521176,6.175313,10.485326,3,3,3,-0.144281,-0.029227
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""LMADELER/2""",16.904433,null,null,null,null,null,null,null,null,NaN,NaN,NaN,0,1,0,NaN,NaN
"""SQGQLNEKPLPEGWEMR/3""",13.006362,null,null,null,null,null,null,null,null,NaN,NaN,NaN,0,1,0,NaN,NaN
"""SGYFQFNK/2""",null,14.248252,null,null,null,null,null,null,null,NaN,NaN,NaN,0,1,0,NaN,NaN


- Because main report and quantification report are related, the annotation and filtering of quantification report can be done by borrowing data from the main report
- The following method will return a new dataframe instead of updating the dataframe stored in the object

In [13]:
report.get_quant_data(
    entry_name="precursor",
    quant_name="precursor_quantity",
    main_df_entry_filter=(
        (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
        & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
    ),
    quant_df_filter=None,
    annotation_cols="protein_group",
)

precursor,20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164,20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163,20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155,20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157,20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220,20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206,20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158,20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210,20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154,H5_Y100_cv_3reps,H25_Y100_cv_3reps,H2d5_Y100_cv_3reps,H5_Y100_detected_reps,H25_Y100_detected_reps,H2d5_Y100_detected_reps,ratio_H25_Y100_to_H5_Y100,ratio_H2d5_Y100_to_H5_Y100,protein_group
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32,str
"""LPAEVEIPKNVDIS/2""",14.434139,13.938183,13.895041,13.805732,14.804245,13.591607,14.452115,14.51086,14.174753,24.786989,29.152855,20.031633,3,3,3,0.145257,-0.382955,"""Q04438"""
"""YFQAPSDYR/2""",13.567189,13.716377,null,null,null,null,null,null,null,NaN,NaN,NaN,0,2,0,NaN,NaN,"""Q08380"""
"""MALHGEGSGEEKGK/3""",19.113167,18.821163,19.034922,18.894535,19.031311,18.908259,19.210786,18.955371,18.854493,11.938686,10.211835,6.492719,3,3,3,-0.03314,-0.092298,"""P23248"""
"""GSQSGPFTYK/2""",14.815426,14.937284,15.032842,14.992521,15.001888,15.263215,14.979494,15.221379,14.561707,9.669373,6.49687,23.621765,3,3,3,-0.148558,-0.087604,"""P38797"""
"""VVDNGSGMC(UniMod:4)K/2""",15.680193,15.928102,null,13.729506,15.545676,11.876769,13.340234,12.41819,11.32639,41.634312,13.666153,NaN,3,3,2,2.466893,-1.632221,"""Q562R1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IFEGTHENPELIWNDNSR/3""",null,null,null,null,13.217183,null,null,null,null,NaN,NaN,NaN,0,1,0,NaN,NaN,"""O75165"""
"""DSVTSPDMDEITHGA/2""",10.174464,null,null,null,null,null,null,null,null,NaN,NaN,NaN,0,1,0,NaN,NaN,"""Q14C86"""
"""TTDGKLPEVTK/2""",null,14.081129,null,null,null,null,null,null,null,NaN,NaN,NaN,0,1,0,NaN,NaN,"""P50454"""


In [14]:
quant_data = report.get_quant_data("precursor", "precursor_quantity")
quant_data.attach_annotation_via_entry(
    "protein_group", persistent=False
)  # Set `persistent` to True will update the dataframe stored in the object
quant_data.filter_entry_by_main_report(
    main_report_filter=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
    & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
)


precursor,20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164,20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163,20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155,20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157,20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220,20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206,20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158,20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210,20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154,H5_Y100_cv_3reps,H25_Y100_cv_3reps,H2d5_Y100_cv_3reps,H5_Y100_detected_reps,H25_Y100_detected_reps,H2d5_Y100_detected_reps,ratio_H25_Y100_to_H5_Y100,ratio_H2d5_Y100_to_H5_Y100
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32
"""LPAEVEIPKNVDIS/2""",14.434139,13.938183,13.895041,13.805732,14.804245,13.591607,14.452115,14.51086,14.174753,24.786989,29.152855,20.031633,3,3,3,0.145257,-0.382955
"""YFQAPSDYR/2""",13.567189,13.716377,null,null,null,null,null,null,null,NaN,NaN,NaN,0,2,0,NaN,NaN
"""MALHGEGSGEEKGK/3""",19.113167,18.821163,19.034922,18.894535,19.031311,18.908259,19.210786,18.955371,18.854493,11.938686,10.211835,6.492719,3,3,3,-0.03314,-0.092298
"""GSQSGPFTYK/2""",14.815426,14.937284,15.032842,14.992521,15.001888,15.263215,14.979494,15.221379,14.561707,9.669373,6.49687,23.621765,3,3,3,-0.148558,-0.087604
"""VVDNGSGMC(UniMod:4)K/2""",15.680193,15.928102,null,13.729506,15.545676,11.876769,13.340234,12.41819,11.32639,41.634312,13.666153,NaN,3,3,2,2.466893,-1.632221
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IFEGTHENPELIWNDNSR/3""",null,null,null,null,13.217183,null,null,null,null,NaN,NaN,NaN,0,1,0,NaN,NaN
"""DSVTSPDMDEITHGA/2""",10.174464,null,null,null,null,null,null,null,null,NaN,NaN,NaN,0,1,0,NaN,NaN
"""TTDGKLPEVTK/2""",null,14.081129,null,null,null,null,null,null,null,NaN,NaN,NaN,0,1,0,NaN,NaN


## Statistics

- lipana has many built-in functions to do different things for statistics, and the statistic pipeline can be customized by chaining built-in functions and customized functions
- Additionally, a packed function has three pre-defined pipelines embedded
- This section will show how to use this function to do tests with any of the three pre-defined pipelines

### Select minimum p-value

- This pipeline will do tests on the base entry level, and each target entry will have the test result selected with the min p-value among its associated base entries
- For example, tests can be done on precursor level, and each protein can have multiple precursors. In this case, for one protein, the FC, p-value, and others will be the same as the one precursor that has min p-value among all precursors that belong to this protein

#### Example: select precursor with min p-value for protein

In [15]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

stats = lipana.do_stats_pipeline_pairwise(
    qdf=report.get_quant_data(entry_name="precursor", quant_name="precursor_quantity").df,
    main_df=report.df,
    design=design,
    target_entry_col="protein_group",
    base_entry_col="precursor",
    group_entry_col=None,
    pipeline="sel_min_p",
    return_chains=False,
)
sleep(0.1)
print("First 5 rows of the result:")
stats.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmp86tb32qj.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmp86tb32qj.txt-limma_output.txt
Quantification file: /tmp/tmphsksx7m6.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmphsksx7m6.txt-limma_output.txt


First 5 rows of the result:


precursor,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,protein_group,row_sign,group_sign,adj_pvalue_exp_wise
str,f64,f64,f64,f64,f64,f64,str,f32,str,f64,f64,f64
"""KQVEQLEK/2""",-15.397284,7.698642,-102.662345,1.0859e-12,7.0257e-10,15.773698,"""H2d5_Y100_vs_H5_Y100""",-15.400756,"""Q14152""",-1.0,-1.0,8.2194e-11
"""ADHQPLTEA/2""",-14.391438,7.195719,-99.930526,1.3191e-12,7.0257e-10,15.728703,"""H2d5_Y100_vs_H5_Y100""",-14.392776,"""P08865""",-1.0,-1.0,8.2194e-11
"""LFPEQQTK/2""",14.187453,7.093726,99.658141,1.3453e-12,7.0257e-10,15.724041,"""H2d5_Y100_vs_H5_Y100""",14.188194,"""P05739""",1.0,1.0,8.2194e-11
"""HLEMNPHFGSHR/3""",-13.921952,6.960976,-97.462274,1.5799e-12,7.0257e-10,15.685227,"""H2d5_Y100_vs_H5_Y100""",-13.922854,"""Q08211""",-1.0,-1.0,8.2194e-11
"""RLEC(UniMod:4)GFPDYK/2""",-13.815381,6.90769,-97.269656,1.6026e-12,7.0257e-10,15.681715,"""H2d5_Y100_vs_H5_Y100""",-13.815996,"""Q3E793""",-1.0,-1.0,8.2194e-11


#### Example: select precursor with min p-value for cut site

- In this case, the report should be first filtered to only keep those non-restricted cut site (i.e. non KR as defined when loading the report)

In [16]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = ~pl.col("cut_site_is_restricted") & (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))

stats = lipana.do_stats_pipeline_pairwise(
    qdf=report.get_quant_data(
        entry_name="precursor",
        quant_name="precursor_quantity",
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    main_df=report.df.filter(filters),
    design=design,
    target_entry_col="cut_site",
    base_entry_col="precursor",
    group_entry_col="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    pipeline="sel_min_p",
    return_chains=False,
)
sleep(0.1)
print("First 5 rows of the result:")
stats.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmp2nucco9b.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmp2nucco9b.txt-limma_output.txt
Quantification file: /tmp/tmp5lq67_2z.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmp5lq67_2z.txt-limma_output.txt


First 5 rows of the result:


precursor,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,cut_site,row_sign,group_sign,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise
str,f64,f64,f64,f64,f64,f64,str,f32,str,f64,f64,f64,str,f64
"""KQVEQLEK/2""",-15.397284,7.698642,-96.296646,3.3869e-13,2.4894e-10,16.855227,"""H2d5_Y100_vs_H5_Y100""",-15.400756,"""Q14152-671_672""",-1.0,-1.0,2.1368e-10,"""Q14152""",6.0964e-12
"""ADHQPLTEA/2""",-14.391438,7.195719,-93.03139,4.4224e-13,2.4894e-10,16.787351,"""H2d5_Y100_vs_H5_Y100""",-14.392776,"""P08865-137_138""",-1.0,-1.0,2.1368e-10,"""P08865""",3.6495e-12
"""LFPEQQTK/2""",14.187453,7.093726,92.570853,4.5954e-13,2.4894e-10,16.777286,"""H2d5_Y100_vs_H5_Y100""",14.188194,"""P05739-120_121""",1.0,1.0,2.1368e-10,"""P05739""",4.5954e-13
"""YDSVEEGEKVVK/2""",-16.438677,8.219339,-89.852936,5.7863e-13,2.4894e-10,16.71522,"""H2d5_Y100_vs_H5_Y100""",-16.452827,"""P51659-72_73""",-1.0,-1.0,2.1368e-10,"""P51659""",2.8931e-12
"""FTPGTFTNQIQAA/2""",-14.546217,7.273109,-89.274668,6.0825e-13,2.4894e-10,16.7014,"""H2d5_Y100_vs_H5_Y100""",-14.550974,"""P08865-115_116""",-1.0,-1.0,2.1368e-10,"""P08865""",3.6495e-12


### Combine p-values

- This pipeline will do tests on base entry level, and use median to aggregate FC values, and Fisher's combined probability test to aggregate p-values
- The api of this method is similar as that for "select min p-value", with only one parameter changed
- Here will only show one example to do tests on peptides and aggregate to cut site 

In [17]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = ~pl.col("cut_site_is_restricted") & (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))

stats = lipana.do_stats_pipeline_pairwise(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_prec",  # this report is generated in one previous cell
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    main_df=report.df.filter(filters),
    design=design,
    target_entry_col="cut_site",
    base_entry_col="stripped_peptide",
    group_entry_col="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    pipeline="combine_p",  # Note this parameter is different from that in the previous example
    return_chains=False,
)
sleep(0.1)
print("First 5 rows of the result:")
stats.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmppkior05g.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmppkior05g.txt-limma_output.txt
Quantification file: /tmp/tmps8gjoksd.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmps8gjoksd.txt-limma_output.txt


First 5 rows of the result:


cut_site,log2_fc,log2_fc_limma,pvalue,adj_pvalue_limma,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,pair
str,f32,f64,f64,f64,f64,str,f64,str
"""P33331-33_34""",0.583285,0.585327,0.007472,0.033623,0.030339,"""P33331""",0.112078,"""H2d5_Y100_vs_H5_Y100"""
"""P13639-652_653""",-1.412413,-1.42665,0.00038,0.002152,0.001907,"""P13639""",0.0005,"""H2d5_Y100_vs_H5_Y100"""
"""P05759-104_105""",-0.072209,-0.118206,0.720269,0.978622,0.964309,"""P05759""",0.977361,"""H2d5_Y100_vs_H5_Y100"""
"""O94927-481_482""",-10.418526,-10.351278,1.2109e-7,8.8120e-7,7.8406e-7,"""O94927""",1.2109e-7,"""H2d5_Y100_vs_H5_Y100"""
"""P02400-73_74""",0.115277,0.108314,0.556856,0.931396,0.905096,"""P02400""",0.953457,"""H2d5_Y100_vs_H5_Y100"""


### Direct tests on target level

- This method will directly do tests on the expected target entry level, and the input should of course be the quantification matrix that is on the target entry level
- Here shows the tests on stripped peptides that are fully-specific

In [18]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = (pl.col("peptide_enzymatic_specificity") == "fully_specific") & (
    ~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)
)

stats = lipana.do_stats_pipeline_pairwise(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_prec",  # this report is generated in one previous cell
        main_df_entry_filter=filters,
    ),
    main_df=report.df.filter(filters),
    design=design,
    target_entry_col="stripped_peptide",
    base_entry_col=None,  # Note in this case, `base_entry_col` should be None
    group_entry_col="protein_group",
    pipeline="direct_test",  # Note this parameter is different from that in the previous example
    return_chains=False,
)
sleep(0.1)
print("First 5 rows of the result:")
stats.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmp65e4al82.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmp65e4al82.txt-limma_output.txt
Quantification file: /tmp/tmpbza64njh.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmpbza64njh.txt-limma_output.txt


First 5 rows of the result:


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise
str,f64,f64,f64,f64,f64,f64,str,f32,f64,str,f64
"""RLECGFPDYK""",-13.815381,6.90769,-104.018898,1.9408e-12,7.6420e-10,15.496262,"""H2d5_Y100_vs_H5_Y100""",-13.815996,7.6420e-10,"""Q3E793""",3.8816e-12
"""EIYHFTLEK""",-14.353165,7.176582,-103.074231,2.0689e-12,7.6420e-10,15.481102,"""H2d5_Y100_vs_H5_Y100""",-14.355913,7.6420e-10,"""Q9BT78""",1.0344e-11
"""QWGWTQGR""",-14.61908,7.30954,-100.38024,2.4906e-12,7.6420e-10,15.435839,"""H2d5_Y100_vs_H5_Y100""",-14.624066,7.6420e-10,"""P18621""",1.4944e-11
"""TDGKVFQFLNAK""",-14.149595,7.074797,-100.225904,2.5176e-12,7.6420e-10,15.433152,"""H2d5_Y100_vs_H5_Y100""",-14.153047,7.6420e-10,"""P83731""",1.0070e-11
"""SADLYKDR""",-13.151591,6.575795,-98.236877,2.8971e-12,7.6420e-10,15.397541,"""H2d5_Y100_vs_H5_Y100""",-13.152547,7.6420e-10,"""Q9HBM1""",2.8971e-12


### Customized pipeline